## Running Conditional VGAE

- Dim-reduction on features ($x \in \mathbb{R}^{N \times P}$: observation; $z \in \mathbb{R}^{N \times D}$: representation; interpretability as "trajectory / gradient")
- Benchmark against K-means clustering & PCA, regular VAE, etc.

In [1]:
import os
import gc
import sys
import pickle
import gzip

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils


In [2]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [3]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, trajectory
import baseline, sb_vgae, model_train, model_eval

from torch_geometric.loader import DataLoader

### VGAE (Xenium-DESI integration)

- Integrate DESI prior to guide latent prob. clustering of Xenium

In [ ]:
import json
import squidpy as sq

from util import IO, utils, gen_graph
from models import  dataset
from torch_geometric import utils as pyg_utils

from importlib import reload
reload(IO)
reload(utils)
reload(gen_graph)
reload(dataset)


In [ ]:
# Load paired Xenium & DESI

xenium_path = '../data/xenium/'
desi_path = '../data/desi_xenium_map/'
n_aux = 5
run_single_sample = True

if run_single_sample:
    sample_id = 'NIH_F5'
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))  # Load xenium
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)  # Load DESI
    adata, adata_desi = IO.filter_cells(adata, adata_desi)  # Filter common cells

    # Dimension reduction on aux. variable
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca']
    # adata.obsm['X_aux'] = adata_desi.X

else:
    # All samples
    sample_ids = sorted([
        f for f in os.listdir(xenium_path) 
        if os.path.isdir(os.path.join(xenium_path, f))
    ])    
    adatas = []
   
    for sample_id in sample_ids:
        print('Loading {}...'.format(sample_id))
        adata = IO.load_xenium(os.path.join(xenium_path, sample_id))  # Load xenium
        adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)  # Load DESI
        adata, adata_desi = IO.filter_cells(adata, adata_desi)  # Filter common cells
    
        # Dimension reduction on aux. variable
        utils.get_PCs(adata_desi, n_pcs=n_aux)
        adata.obsm['X_aux'] = adata_desi.obsm['X_pca']
    
        adatas.append(adata)
        del adata_desi

gc.collect()


#### Model sketch iVAE: 
- $z_i \mid u_i \sim \mathcal{N}(f_{\lambda_{\mu}}(u_i), f_{\lambda_{\sigma}}(u_i))$ vs. $z_i \mid u_i \sim \mathcal{vMF}(f_{\lambda}(u_i))$
- $x_i \mid z_i, \mathbf{A} \sim \mathcal{NegBinom}(l \cdot f_{\theta}(z_i, \mathbf{A}, \theta_g)$

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pyro

from torch_geometric.data import DataLoader
from models import vgae, model_train

In [ ]:
if run_single_sample:
    loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=16)
    graph_data = loader.load_graphs([adata])
    train_dl = DataLoader(graph_data, shuffle=True)
else:
    # Randomly split samples for training / test (per gender)
    split_ratio = 0.6
    female_indices = np.arange(len(adatas)//2)
    male_indices = np.arange(len(adatas)//2, len(adatas))
    
    train_indices = list(np.random.choice(female_indices, size=int(len(female_indices)*split_ratio), replace=False)) + \
                    list(np.random.choice(male_indices, size=int(len(male_indices)*split_ratio), replace=False))

    loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=16)
    train_data = loader.load_graphs([adatas[i] for i in train_indices])
    train_dl = DataLoader(train_data, batch_size=16, shuffle=True)

gc.collect()

In [ ]:
from importlib import reload

reload(IO)
reload(utils)
reload(plot)
reload(configs)
reload(vgae)
reload(model_train)

torch.manual_seed(0)

In [ ]:
lr = 1e-3
device = torch.device('cuda')
# prior_choice='vMF'
prior_choice = 'normal'

n_epochs = 100
n_hidden = 64
n_latent = 8
n_aux = adata.obsm['X_aux'].shape[1]

train_configs = configs.set_train_configs(
    n_epochs=n_epochs, 
    lr=lr, gamma=1.0,
    annealing=False,
    device=device
)

model_configs = configs.set_model_configs(
    prior=prior_choice, act=nn.SiLU(), 
    c_in=adata.shape[1], c_aux=n_aux, c_hidden=n_hidden, c_latent=n_latent, c_embedding=16,
    beta=0.5, k_hop=2, dropout=0.5,
    device=device, enc_option='attn', num_heads=1
) 

In [ ]:
pyro.clear_param_store()
model = vgae.VGAE(model_configs, device=train_configs.device)
model, losses = model_train.train_vgae(model, train_dl, train_configs)

In [ ]:
plt.figure(figsize=(5, 2))
plt.plot(np.arange(train_configs.n_epochs), losses)
plt.xlabel('Epochs')
plt.ylabel('Train ELBO')
plt.show()

In [ ]:
# Archive: check cell-type specific reconstruction

n_cells = adata.shape[0]
idx = np.arange(n_cells)
np.random.shuffle(idx)
idx = idx[:500]
coords = adata[:500].obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
G = gen_graph.construct_graph(coords, k=30, r=np.inf)
edge_index = pyg_utils.from_networkx(G).edge_index
del G

model.device = 'cpu'
model = model.to('cpu')
pred_params = model.predict(
    adata.X.A, 
    adata.obsm['X_aux'], 
    edge_index
)
px = pred_params['px_mu'].detach().cpu().numpy()

# annot = IO.load_xenium(os.path.join(xenium_path, sample_id), raw_count=False).obs['cell_type']
# annot = annot[adata.obs_names]

# for cell_type in np.unique(annot):
#     x_true = adata[annot[annot == cell_type].index].X.A
#     barcodes = adata[annot[annot == cell_type].index].obs_names
#     cell_indices = np.array([
#         i for i, label in enumerate(adata.obs_names)
#         if label in barcodes
#     ])
#     x_pred = px[cell_indices]

#     assert len(x_true) == len(x_pred)

#     plt.figure(figsize=(5, 5))
#     plt.scatter(x_true.flatten(), x_pred.flatten(), s=0.5, alpha=0.5)
#     plt.suptitle(cell_type)
#     plt.show()

# del cell_type, x_true, x_pred, barcodes, cell_indices

x_true = adata.X.A
plt.figure(figsize=(4, 4))
plt.scatter(x_true.flatten(), px.flatten(), s=.5)
plt.show()

In [ ]:
sns.heatmap(pred_params['qz_params'][-1].mean(0).detach().numpy())
plt.show()

In [ ]:
if run_single_sample:
    
    import warnings
    warnings.filterwarnings('ignore')
    import scFates as scf
    import igraph
    
    n_cells = adata.shape[0]
    coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
    G = gen_graph.construct_graph(coords, k=30, r=np.inf)
    edge_index = pyg_utils.from_networkx(G).edge_index
    del G
    
    model.device = 'cpu'
    model = model.to('cpu')
    pred_params = model.predict(
        adata.X.A, 
        adata.obsm['X_aux'], 
        edge_index
    )
    qz = pred_params.qz_params[0].detach().cpu().numpy()
    px = pred_params['px_mu'].detach().cpu().numpy()
    adata.obsm['X_z'] = qz

    # Check reconstruction quality
    indices = np.random.choice(
        adata.shape[1], size=300, replace=False
    )
    plt.figure(figsize=(6, 4))
    plt.scatter(adata.X.A[:, indices].flatten(), px[:, indices].flatten(), s=.5)
    plt.xlabel('X')
    plt.ylabel('X_reconst')
    plt.show()

    # Trajectory inference
    scf.tl.curve(
        adata,
        use_rep='X_z',
        Nodes=n_latent, 
        epg_extend_leaves=True,
        ndims_rep=n_latent
    )

    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        dist_metric=prior_choice, 
        k=0
    )

    sc.pp.neighbors(adata, use_rep='X_z')
    sc.tl.umap(adata)

    # Spatial visualization
    display_trajectory = True
    if display_trajectory:
        scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent))) 
    
        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            title='Pseudotime\n'+'LYNX'
        )
        
        sq.pl.spatial_scatter(
            adata, color='milestones', 
            cmap='tab10', size=20, img=False,
            title='Zonation\n'+'LYNX'
        )

In [31]:
torch.save(model.state_dict(), os.path.join('../results/', '{}_multi.pt').format(model_configs.prior))

In [ ]:
del adatas, train_data, train_dl
gc.collect()

#### Held-out validation - metrics

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import scFates as scf
import igraph

import json
import squidpy as sq

import pyro
from pyro.infer import SVI, Trace_ELBO, infer_discrete

from util import IO, utils, gen_graph, trajectory
from models import vgae, model_train, dataset
from torch_geometric import utils as pyg_utils
from torch_geometric.data import DataLoader

from importlib import reload
reload(utils)
reload(gen_graph)
reload(dataset)
reload(trajectory)

In [34]:
def evaluate_elbo(model, dataloader, device=torch.device('cpu')):
    from pyro.optim import Adam
    optimizer = Adam({"lr": 1.0e-3})  # dummy optimizer
    
    model = model.to(device)
    elbo = Trace_ELBO()
    svi = SVI(model.model, model.guide, optimizer, elbo)
    elbos = []
    
    for data in dataloader:
        x = data.x.to(device).float()
        u = data.u.to(device).float()
        edge_index = data.edge_index.to(device)
        elbo = svi.evaluate_loss(x, u, edge_index)

        elbos.append(elbo / x.shape[0])

    return elbos

def evaluate_kl(model, dataloader, device=torch.device('cpu')):
    from pyro.optim import Adam
    from pyro.infer import TraceMeanField_ELBO
    from pyro import poutine
    
    optimizer = Adam({"lr": 1.0e-3})  # dummy optimizer
    
    model = model.to(device)
    elbo = TraceMeanField_ELBO()
    svi = SVI(model.model, model.guide, optimizer, elbo)
    
    kl_divs = []
    for data in dataloader:
        x = data.x.to(device).float()
        u = data.u.to(device).float()
        edge_index = data.edge_index.to(device)

        with poutine.scale(scale=1e-7):
            kl_div = svi.evaluate_loss(x, u, edge_index)
            kl_divs.append(kl_div)

    return kl_divs

def cell_type_dynamics(
    adata,
    cell_type, 
    show_fig=False
):
    """
    Dynamics of cell-type proportion along discrete zonations
    """
    assert 'milestones' in adata.obs.columns and 'cell_type' in adata.obs.columns
    assert cell_type in adata.obs['cell_type'].unique()

    zonation_states = np.unique(adata.obs['milestones'])
    dynamics = []
    for state in zonation_states:
        summary = adata[adata.obs['milestones'] == state].obs['cell_type'].value_counts()
        if cell_type in summary.index:
            target_count = summary[cell_type]
            total_count = summary.sum()
            dynamics.append(target_count / total_count)
        else:
            dynamics.append(0)

    if show_fig:
        fig, ax = plt.subplots(figsize=(4, 1))
        ax.plot(zonation_states, dynamics, '.--', c='blue')
        ax.set_title(cell_type)
        plt.show()
    
    return dynamics

In [ ]:
# Load validation data
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
n_aux = 10
n_latent = 8
n_hidden = 16

sample_ids = np.array(sorted([
    f for f in os.listdir(xenium_path) 
    if os.path.isdir(os.path.join(xenium_path, f))
]))

train_ids = np.array(sample_ids[train_indices])
val_ids = np.setdiff1d(np.arange(len(sample_ids)), train_indices)

adatas_train = []
for sample_id in train_ids:
    print('Loading training sample {}...'.format(sample_id))
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    adata, adata_desi = IO.filter_cells(adata, adata_desi)
    
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)
    
    adatas_train.append(adata)
    del adata_desi
    gc.collect()
    
adatas_val = []
for sample_id in val_ids:
    print('Loading val sample {}...'.format(sample_id))
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    adata, adata_desi = IO.filter_cells(adata, adata_desi)
    
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)
    
    adatas_val.append(adata)
    del adata_desi
    gc.collect()

del sample_id

In [ ]:
# Load models
prior_choice = 'normal'
device = torch.device('cuda')
model_configs = configs.set_model_configs(
    device=device, prior=prior_choice, act=nn.SiLU(), beta=0.5, 
    c_in=adata.shape[1], c_aux=n_aux, c_hidden=8, c_latent=n_latent,
    k_hop=3, dropout=0.5
) 

model_normal = vgae.VGAE(configs=model_configs, device=device)
model_normal.load_state_dict(torch.load('../results/{}_multi.pt'.format(prior_choice)))

prior_choice = 'vMF'
model_configs = configs.set_model_configs(
    device=device, prior=prior_choice, act=nn.SiLU(), beta=0.5, 
    c_in=adata.shape[1], c_aux=n_aux, c_hidden=8, c_latent=n_latent,
    k_hop=3, dropout=0.5
) 

model_vMF = vgae.VGAE(configs=model_configs, device=device)
model_vMF.load_state_dict(torch.load('../results/{}_multi.pt'.format(prior_choice)))

Fitting metrics:
- Neg. ELBO
- Reconstruction NLL
- KL div.

In [ ]:
# Create subgraph-batched graph data
loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=8)
train_data = loader.load_graphs(adatas_train)
val_data = loader.load_graphs(adatas_val)
train_dl = DataLoader(train_data)
val_dl = DataLoader(val_data)

del train_data, val_data
gc.collect()

In [ ]:
train_elbos_normal = evaluate_elbo(model_normal, train_dl, device=device)
val_elbos_normal = evaluate_elbo(model_normal, val_dl, device=device)

train_elbos_vMF = evaluate_elbo(model_vMF, train_dl, device=device)
val_elbos_vMF = evaluate_elbo(model_vMF, val_dl, device=device)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
elbo_df = pd.DataFrame({
    'ELBO': train_elbos_normal + train_elbos_vMF + \
            val_elbos_normal + val_elbos_vMF,
    
    'Label': ['train']*len(train_elbos_normal)*2 + \
             ['val']*len(val_elbos_normal)*2,
    
    'Prior': ['Normal']*len(train_elbos_normal) + \
             ['vMF']*len(train_elbos_vMF) + \
             ['Normal']*len(val_elbos_normal) + \
             ['vMF']*len(val_elbos_vMF)
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.barplot(x='Label', y='ELBO', data=elbo_df, hue='Prior')
ax.spines[['right', 'top']].set_visible(False)
sns.move_legend(ax, "upper right", bbox_to_anchor=(1.2, 0.7))
ax.set_title('ELBO')
plt.show()

In [ ]:
train_kls_normal = evaluate_kl(model_normal, train_dl, device=device)
val_kls_normal = evaluate_kl(model_normal, val_dl, device=device)

train_kls_vMF = evaluate_kl(model_vMF, train_dl, device=device)
val_kls_vMF = evaluate_kl(model_vMF, val_dl, device=device)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
kl_df = pd.DataFrame({
    'KL': train_kls_normal + train_kls_vMF + \
            val_kls_normal + val_kls_vMF,
    
    'Label': ['train']*len(train_kls_normal)*2 + \
             ['val']*len(val_kls_normal)*2,
    
    'Prior': ['Normal']*len(train_kls_normal) + \
             ['vMF']*len(train_kls_vMF) + \
             ['Normal']*len(val_kls_normal) + \
             ['vMF']*len(val_kls_vMF)
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.barplot(x='Label', y='KL', data=kl_df, hue='Prior')
ax.spines[['right', 'top']].set_visible(False)
sns.move_legend(ax, "upper right", bbox_to_anchor=(1.2, 0.7))
ax.set_title('KL divergence')
plt.show()

In [ ]:
from scipy.stats import linregress

train_r2s_normal = []
train_r2s_vMF = []

for i, adata in enumerate(adatas_train):
    print('Evaluating on train sample {}...'.format(i))
    coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
    G = gen_graph.construct_graph(coords, 
                                  k=30, 
                                  r=np.inf)
    edge_index = pyg_utils.from_networkx(G).edge_index
    del G
    
    model_normal.device = 'cpu'
    model_normal = model_normal.to('cpu')
    pred_params = model_normal.predict(adata.X.A, 
                                       adata.obsm['X_aux'], 
                                       edge_index)
    px = pred_params.px.detach().cpu().numpy()

    _, _, r_val, _, _ = linregress(adata.X.A.flatten(), px.flatten())
    train_r2s_normal.append(r_val**2)
    
    model_vMF.device = 'cpu'
    model_vMF = model_vMF.to('cpu')
    pred_params = model_vMF.predict(adata.X.A, 
                                    adata.obsm['X_aux'], 
                                    edge_index)
    px = pred_params.px.detach().cpu().numpy()

    _, _, r_val, _, _ = linregress(adata.X.A.flatten(), px.flatten())
    train_r2s_vMF.append(r_val**2)

    del edge_index, adata
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
val_r2s_normal = []
val_r2s_vMF = []

for i, adata in enumerate(adatas_val):
    print('Evaluating on val sample {}...'.format(i))
    coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
    G = gen_graph.construct_graph(coords, 
                                  k=30, 
                                  r=np.inf)
    edge_index = pyg_utils.from_networkx(G).edge_index
    del G
    
    model_normal.device = 'cpu'
    model_normal = model_normal.to('cpu')
    pred_params = model_normal.predict(adata.X.A, 
                                       adata.obsm['X_aux'], 
                                       edge_index)
    px = pred_params.px.detach().cpu().numpy()

    _, _, r_val, _, _ = linregress(adata.X.A.flatten(), px.flatten())
    val_r2s_normal.append(r_val**2)
    
    model_vMF.device = 'cpu'
    model_vMF = model_vMF.to('cpu')
    pred_params = model_vMF.predict(adata.X.A, 
                                    adata.obsm['X_aux'], 
                                    edge_index)
    px = pred_params.px.detach().cpu().numpy()

    _, _, r_val, _, _ = linregress(adata.X.A.flatten(), px.flatten())
    val_r2s_vMF.append(r_val**2)

    del edge_index, adata
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
recon_df = pd.DataFrame({
    r'$R^2$': train_r2s_normal + train_r2s_vMF + \
             val_r2s_normal + val_r2s_vMF,
    
    'Label': ['train']*len(train_r2s_normal)*2 + \
             ['val']*len(val_r2s_normal)*2,
    
    'Prior': ['Normal']*len(train_r2s_normal) + \
             ['vMF']*len(train_r2s_vMF) + \
             ['Normal']*len(val_r2s_normal) + \
             ['vMF']*len(val_r2s_vMF)
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.barplot(x='Label', y=r'$R^2$', data=recon_df, hue='Prior')
ax.spines[['right', 'top']].set_visible(False)
sns.move_legend(ax, "upper right", bbox_to_anchor=(1.2, 0.7))
ax.set_title('Reconstruction fitting')
plt.show()

In [ ]:
# Visualize disentanglement btw latent features

# sns.clustermap(np.corrcoef(qz.T), cmap='coolwarm')
# plot.disp_spatial_latents(adata, qz, ncols=4)

In [ ]:
# rand_indices = np.random.choice(np.arange(adatas_val[-1].X.A.shape[1]), size=100, replace=False)

# plt.figure(figsize=(6, 4))
# plt.scatter(adatas_val[-1].X.A[:, rand_indices].flatten(), px[:, rand_indices].flatten(), s=1)
# plt.xlabel('X')
# plt.ylabel("X_reconst")
# plt.show()

# del rand_indices

#### Held-out validation - antibody

In [44]:
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'

sample_ids = np.array(sorted([
    f for f in os.listdir(xenium_path) 
    if os.path.isdir(os.path.join(xenium_path, f))
]))

train_ids = sample_ids[train_indices]
val_ids = sample_ids[np.setdiff1d(np.arange(len(sample_ids)), train_indices)]


In [ ]:
# Load held-out data
adatas_val = []
for sample_id in val_ids:
    print('Loading val sample {}...'.format(sample_id))
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    adata, adata_desi = IO.filter_cells(adata, adata_desi)
    
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)
    
    adatas_val.append(adata)
    del adata_desi
    gc.collect()

del sample_id

In [ ]:
adatas_val[0].shape[1]

In [ ]:
# Load model
n_latent = 8
n_aux = 10
n_hidden = 16
n_features = adatas_val[0].shape[1]
device = torch.device('cpu')

model_configs = configs.set_model_configs(
    device=device, prior=prior_choice, act=nn.SiLU(), beta=0.5, 
    c_in=n_features, c_aux=n_aux, c_hidden=n_hidden, c_latent=n_latent,
    k_hop=3, dropout=0.5
) 

pyro.clear_param_store()
model = vgae.VGAE(model_configs, device=train_configs.device)
model.load_state_dict(torch.load('../results/vMF_multi.pt'))

In [ ]:
display = True

for i, sample_id in enumerate(val_ids):
    # ----------------------------------------
    print('Visual checks on trajectory of {}...'.format(sample_id))
    print('\t Computing latent representation...')
    
    adata = adatas_val[i]
    n_cells = adata.shape[0]
    coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
    G = gen_graph.construct_graph(coords, k=30, r=np.inf)
    edge_index = pyg_utils.from_networkx(G).edge_index
    del G

    model.device = 'cpu'
    model = model.to('cpu')

    pred_params = model.predict(
        adata.X.A, 
        adata.obsm['X_aux'], 
        edge_index
    )
    
    qz = utils.pnorm(pred_params.qz_params.detach().cpu().numpy())  # Important: project to hypersphere
    px = pred_params.px.detach().cpu().numpy()
    adata.obsm['X_z'] = qz
    
    scf.tl.curve(
        adata, 
        use_rep='X_z',
        Nodes=n_latent, 
        epg_extend_leaves=True,
        ndims_rep=n_latent
    )

    trajectory.compute_trajectory(
        adata, 
        use_rep='X_z',
        dist_metric='vMF', 
        k=0
    )

    if display:
        # Reconstruction quality
        indices = np.random.choice(
            adata.shape[1], size=100, replace=False
        )
        plt.figure(figsize=(6, 4))
        plt.scatter(adata.X.A[:, indices].flatten(), px[:, indices].flatten(), s=.5)
        plt.xlabel('X')
        plt.ylabel('X_reconst')
        plt.show()

        # Latent representation
        sc.pp.neighbors(adata, use_rep='X_z')
        sc.tl.umap(adata)
    
        scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent)))
        sc.pl.umap(
            adata, color='t',
            vmin=np.quantile(adata.obs['t'], .1),
            vmax=np.quantile(adata.obs['t'], .9),
            cmap='RdBu'
        )

        sq.pl.spatial_scatter(
            adata, color='t', 
            cmap='RdBu', size=20, img=False,
            vmin=np.quantile(adata.obs['t'], .1),
            vmax=np.quantile(adata.obs['t'], .9),
            title='Pseudotime\n (vMF Prior)'
        )
        
        sq.pl.spatial_scatter(
            adata, color='milestones', 
            cmap='tab20', size=20, img=False,
            title='Zonation\n (vMF Prior)'
        )
    
    # ----------------------------------------
    print('\t Loading paired Ab images...')
    filename = '../data/integrated/antibody/{}.ome.tif'.format(sample_id)
    img = tifffile.imread(filename)[1:]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
    ax1.imshow(img[1], cmap='magma')
    ax1.set_title('Peri-central')
    ax1.axis('off')

    ax2.imshow(img[2], cmap='magma')
    ax2.set_title('Peri-portal')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

    # Save latent
    np.save('../results/lynx_{0}_{1}.npy'.format(sample_id, n_latent), qz)
    
    print('====================================\n\n')
    del adata, img
    gc.collect()


In [ ]:
np.mean(px)

**[DEBUG]**
- check patient-specific reconstruction loss
- check marker-specific reconstruction loss
- add covariates to multi-patient training

Check covariate effects (zonation interpretation flipped on `NIH_F3` - BME ~2X as others)

In [ ]:
sample_id = 'NIH_M4'

adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
adata, adata_desi = IO.filter_cells(adata, adata_desi)

coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
G = gen_graph.construct_graph(coords, k=30, r=np.inf)
edge_index = pyg_utils.from_networkx(G).edge_index
del coords, G
gc.collect()

utils.get_PCs(adata_desi, n_pcs=n_aux)
adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)
del adata_desi

pred_params = model.predict(
    adata.X.A, 
    adata.obsm['X_aux'], 
    edge_index
)
qz = utils.pnorm(pred_params.qz_params.detach().cpu().numpy())  # Important: project to hypersphere
px = pred_params.px.detach().cpu().numpy()
adata.obsm['X_z'] = qz

scf.tl.curve(
    adata, 
    use_rep='X_z',
    Nodes=n_latent, 
    epg_extend_leaves=True,
    ndims_rep=n_latent
)

trajectory.compute_trajectory(
    adata, 
    use_rep='X_z',
    dist_metric='vMF', 
    k=0
)

gc.collect()

In [ ]:
# Latent representation
sc.pp.neighbors(adata, use_rep='X_z')
sc.tl.umap(adata)

scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent)))
sc.pl.umap(
    adata, color='t',
    vmin=np.quantile(adata.obs['t'], .1),
    vmax=np.quantile(adata.obs['t'], .9),
    cmap='RdBu'
)

sq.pl.spatial_scatter(
    adata, color='t', 
    cmap='RdBu', size=20, img=False,
    vmin=np.quantile(adata.obs['t'], .1),
    vmax=np.quantile(adata.obs['t'], .9),
    title='Pseudotime\n (vMF Prior)'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab20', size=20, img=False,
    title='Zonation\n (vMF Prior)'
)

In [ ]:
filename = '../data/integrated/antibody/{}.ome.tif'.format(sample_id)
img = tifffile.imread(filename)[1:]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
ax1.imshow(img[1], cmap='magma')
ax1.set_title('Peri-central')
ax1.axis('off')

ax2.imshow(img[2], cmap='magma')
ax2.set_title('Peri-portal')
ax2.axis('off')

plt.tight_layout()
plt.show()


---